# E20260909125807233308: E4 — QLoRA A/B consistency

Контролируемая абляция поверх E2: тот же DeBERTa-v3-base QLoRA, split, seed и random A/B swap; единственное изменение — $\lambda=0.1$ Jensen–Shannon penalty между двумя порядками ответов.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "configs/project.json").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Откройте notebook внутри клонированного репозитория.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from pmldl_llm.e4_consistency import run_e4_consistency_experiment
from pmldl_llm.notebook import load_experiment_setup, run_notebook_experiment

PROJECT_ROOT

## Frozen experiment contract

Конфиг фиксирует E2 control, selection fold и все гиперпараметры. Первый запуск обязан быть smoke-test.

In [ ]:
SETUP = load_experiment_setup(
    "configs/experiments/E20260909125807233308.json",
    project_root=PROJECT_ROOT,
)
SETUP

## Train and evaluate

Runner проверяет frozen data checksums и fold roles, пишет локальные артефакты и отправляет live-метрики в ClearML.

In [ ]:
def train_and_evaluate(run):
    return run_e4_consistency_experiment(run, SETUP, PROJECT_ROOT)

In [ ]:
RESULT = run_notebook_experiment(
    train_and_evaluate,
    SETUP,
    project_root=PROJECT_ROOT,
)
RESULT

## Acceptance gates

После успешного smoke-run: импортировать его артефакты, выполнить `make prepare-full EXPERIMENT=E20260909125807233308` и только затем запускать full-run. Сравниваем selection log loss и raw swap error с зафиксированным E2 QLoRA control.